In [2]:
#Step 1: Import Libraries

import pandas as pd
import numpy as np
import string
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

In [8]:
#Step 2: Load Your Dataset
df = pd.read_csv("dataset_nlp.csv", encoding='latin-1', engine='python', on_bad_lines='skip')
df.head()

,id,asins,brand,categories,colors,dateAdded,dateUpdated,dimension,ean,keys,...,reviews.rating,reviews.sourceURLs,reviews.text,reviews.title,reviews.userCity,reviews.userProvince,reviews.username,sizes,upc,weight
0,AVpe7AsMilAPnD_xQ78G,B00QJDU3KY,Amazon,"Amazon Devices,mazon.co.uk",NaN,2016-03-08T20:21:53Z,2017-07-18T23:52:58Z,169 mm x 117 mm x 9.1 mm,NaN,kindlepaperwhite/b00qjdu3ky,...,5.0,https://www.amazon.com/Kindle-Paperwhite-High-...,I initially had trouble deciding between the p...,"Paperwhite voyage, no regrets!",NaN,NaN,Cristina M,NaN,NaN,205 grams
1,AVpe7AsMilAPnD_xQ78G,B00QJDU3KY,Amazon,"Amazon Devices,mazon.co.uk",NaN,2016-03-08T20:21:53Z,2017-07-18T23:52:58Z,169 mm x 117 mm x 9.1 mm,NaN,kindlepaperwhite/b00qjdu3ky,...,5.0,https://www.amazon.com/Kindle-Paperwhite-High-...,Allow me to preface this with a little history...,One Simply Could Not Ask For More,NaN,NaN,Ricky,NaN,NaN,205 grams
2,AVpe7AsMilAPnD_xQ78G,B00QJDU3KY,Amazon,"Amazon Devices,mazon.co.uk",NaN,2016-03-08T20:21:53Z,2017-07-18T23:52:58Z,169 mm x 117 mm x 9.1 mm,NaN,kindlepaperwhite/b00qjdu3ky,...,4.0,https://www.amazon.com/Kindle-Paperwhite-High-...,I am enjoying it so far. Great for reading. Ha...,Great for those that just want an e-reader,NaN,NaN,Tedd Gardiner,NaN,NaN,205 grams
3,AVpe7AsMilAPnD_xQ78G,B00QJDU3KY,Amazon,"Amazon Devices,mazon.co.uk",NaN,2016-03-08T20:21:53Z,2017-07-18T23:52:58Z,169 mm x 117 mm x 9.1 mm,NaN,kindlepaperwhite/b00qjdu3ky,...,5.0,https://www.amazon.com/Kindle-Paperwhite-High-...,I bought one of the first Paperwhites and have...,Love / Hate relationship,NaN,NaN,Dougal,NaN,NaN,205 grams
4,AVpe7AsMilAPnD_xQ78G,B00QJDU3KY,Amazon,"Amazon Devices,mazon.co.uk",NaN,2016-03-08T20:21:53Z,2017-07-18T23:52:58Z,169 mm x 117 mm x 9.1 mm,NaN,kindlepaperwhite/b00qjdu3ky,...,5.0,https://www.amazon.com/Kindle-Paperwhite-High-...,I have to say upfront - I don't like coroporat...,I LOVE IT,NaN,NaN,Miljan David Tanic,NaN,NaN,205 grams


In [9]:
df.info()
df.columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1343 entries, 0 to 1342
Data columns (total 27 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    1343 non-null   object 
 1   asins                 1343 non-null   object 
 2   brand                 1343 non-null   object 
 3   categories            1343 non-null   object 
 4   colors                578 non-null    object 
 5   dateAdded             1343 non-null   object 
 6   dateUpdated           1343 non-null   object 
 7   dimension             514 non-null    object 
 8   ean                   714 non-null    float64
 9   keys                  1343 non-null   object 
 10  manufacturer          769 non-null    object 
 11  manufacturerNumber    689 non-null    object 
 12  name                  1343 non-null   object 
 13  prices                1343 non-null   object 
 14  reviews.date          1039 non-null   object 
 15  reviews.doRecommend  

Index(['id', 'asins', 'brand', 'categories', 'colors', 'dateAdded',
       'dateUpdated', 'dimension', 'ean', 'keys', 'manufacturer',
       'manufacturerNumber', 'name', 'prices', 'reviews.date',
       'reviews.doRecommend', 'reviews.numHelpful', 'reviews.rating',
       'reviews.sourceURLs', 'reviews.text', 'reviews.title',
       'reviews.userCity', 'reviews.userProvince', 'reviews.username', 'sizes',
       'upc', 'weight'],
      dtype='object')

In [10]:
#Step 4: Select Required Columns
df = df[['reviews.text', 'reviews.rating']]
df.head()

,reviews.text,reviews.rating
0,I initially had trouble deciding between the p...,5.0
1,Allow me to preface this with a little history...,5.0
2,I am enjoying it so far. Great for reading. Ha...,4.0
3,I bought one of the first Paperwhites and have...,5.0
4,I have to say upfront - I don't like coroporat...,5.0


In [11]:
#Step 5: Remove Missing Values
df = df.dropna()

In [12]:
#Step 6: Create Labels
# Remove neutral (rating = 3)
df = df[df['reviews.rating'] != 3]

# Create sentiment label
df['label'] = df['reviews.rating'].apply(lambda x: 1 if x >= 4 else 0)

df.head()

,reviews.text,reviews.rating,label
0,I initially had trouble deciding between the p...,5.0,1
1,Allow me to preface this with a little history...,5.0,1
2,I am enjoying it so far. Great for reading. Ha...,4.0,1
3,I bought one of the first Paperwhites and have...,5.0,1
4,I have to say upfront - I don't like coroporat...,5.0,1


In [30]:
#Step 7: Rename Text Column
df = df.rename(columns={'reviews.text': 'text'})
df.head()

,text,reviews.rating,label,tokens,processed_text
0,i initially had trouble deciding between the p...,5.0,1,"[initi, troubl, decid, paperwhit, voyag, revie...",initi troubl decid paperwhit voyag review less...
1,allow me to preface this with a little history...,5.0,1,"[allow, prefac, littl, histori, casual, reader...",allow prefac littl histori casual reader own n...
2,i am enjoying it so far great for reading had ...,4.0,1,"[enjoy, far, great, read, origin, fire, sinc, ...",enjoy far great read origin fire sinc 2012 fir...
3,i bought one of the first paperwhites and have...,5.0,1,"[bought, one, first, paperwhit, plea, constant...",bought one first paperwhit plea constant compa...
4,i have to say upfront i dont like coroporate ...,5.0,1,"[say, upfront, dont, like, coropor, hermet, cl...",say upfront dont like coropor hermet close stu...


In [31]:
#Step 8: Preprocessing
df['text'] = df['text'].str.lower()
df.head()

,text,reviews.rating,label,tokens,processed_text
0,i initially had trouble deciding between the p...,5.0,1,"[initi, troubl, decid, paperwhit, voyag, revie...",initi troubl decid paperwhit voyag review less...
1,allow me to preface this with a little history...,5.0,1,"[allow, prefac, littl, histori, casual, reader...",allow prefac littl histori casual reader own n...
2,i am enjoying it so far great for reading had ...,4.0,1,"[enjoy, far, great, read, origin, fire, sinc, ...",enjoy far great read origin fire sinc 2012 fir...
3,i bought one of the first paperwhites and have...,5.0,1,"[bought, one, first, paperwhit, plea, constant...",bought one first paperwhit plea constant compa...
4,i have to say upfront i dont like coroporate ...,5.0,1,"[say, upfront, dont, like, coropor, hermet, cl...",say upfront dont like coropor hermet close stu...


In [32]:
#Remove Punctuation
import string
df['text'] = df['text'].apply(lambda x: ''.join([c for c in x if c not in string.punctuation]))
df.head()

,text,reviews.rating,label,tokens,processed_text
0,i initially had trouble deciding between the p...,5.0,1,"[initi, troubl, decid, paperwhit, voyag, revie...",initi troubl decid paperwhit voyag review less...
1,allow me to preface this with a little history...,5.0,1,"[allow, prefac, littl, histori, casual, reader...",allow prefac littl histori casual reader own n...
2,i am enjoying it so far great for reading had ...,4.0,1,"[enjoy, far, great, read, origin, fire, sinc, ...",enjoy far great read origin fire sinc 2012 fir...
3,i bought one of the first paperwhites and have...,5.0,1,"[bought, one, first, paperwhit, plea, constant...",bought one first paperwhit plea constant compa...
4,i have to say upfront i dont like coroporate ...,5.0,1,"[say, upfront, dont, like, coropor, hermet, cl...",say upfront dont like coropor hermet close stu...


In [33]:
#Tokenization
df['tokens'] = df['text'].apply(lambda x: x.split())
df.head()

,text,reviews.rating,label,tokens,processed_text
0,i initially had trouble deciding between the p...,5.0,1,"[i, initially, had, trouble, deciding, between...",initi troubl decid paperwhit voyag review less...
1,allow me to preface this with a little history...,5.0,1,"[allow, me, to, preface, this, with, a, little...",allow prefac littl histori casual reader own n...
2,i am enjoying it so far great for reading had ...,4.0,1,"[i, am, enjoying, it, so, far, great, for, rea...",enjoy far great read origin fire sinc 2012 fir...
3,i bought one of the first paperwhites and have...,5.0,1,"[i, bought, one, of, the, first, paperwhites, ...",bought one first paperwhit plea constant compa...
4,i have to say upfront i dont like coroporate ...,5.0,1,"[i, have, to, say, upfront, i, dont, like, cor...",say upfront dont like coropor hermet close stu...


In [18]:
#Stopwords Removal
import nltk

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
import nltk
nltk.download('all')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/abc.zip.
[nltk_data]    | Downloading package alpino to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/alpino.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping
[nltk_data]    |       taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     /root/nl

In [34]:
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

df['tokens'] = df['tokens'].apply(lambda words: [w for w in words if w not in stop_words])
df.head()

,text,reviews.rating,label,tokens,processed_text
0,i initially had trouble deciding between the p...,5.0,1,"[initially, trouble, deciding, paperwhite, voy...",initi troubl decid paperwhit voyag review less...
1,allow me to preface this with a little history...,5.0,1,"[allow, preface, little, history, casual, read...",allow prefac littl histori casual reader own n...
2,i am enjoying it so far great for reading had ...,4.0,1,"[enjoying, far, great, reading, original, fire...",enjoy far great read origin fire sinc 2012 fir...
3,i bought one of the first paperwhites and have...,5.0,1,"[bought, one, first, paperwhites, pleased, con...",bought one first paperwhit plea constant compa...
4,i have to say upfront i dont like coroporate ...,5.0,1,"[say, upfront, dont, like, coroporate, hermeti...",say upfront dont like coropor hermet close stu...


In [35]:
#Stemming
from nltk.stem import PorterStemmer
stemmer = PorterStemmer()

df['tokens'] = df['tokens'].apply(lambda words: [stemmer.stem(w) for w in words])
df.head()

,text,reviews.rating,label,tokens,processed_text
0,i initially had trouble deciding between the p...,5.0,1,"[initi, troubl, decid, paperwhit, voyag, revie...",initi troubl decid paperwhit voyag review less...
1,allow me to preface this with a little history...,5.0,1,"[allow, prefac, littl, histori, casual, reader...",allow prefac littl histori casual reader own n...
2,i am enjoying it so far great for reading had ...,4.0,1,"[enjoy, far, great, read, origin, fire, sinc, ...",enjoy far great read origin fire sinc 2012 fir...
3,i bought one of the first paperwhites and have...,5.0,1,"[bought, one, first, paperwhit, pleas, constan...",bought one first paperwhit plea constant compa...
4,i have to say upfront i dont like coroporate ...,5.0,1,"[say, upfront, dont, like, coropor, hermet, cl...",say upfront dont like coropor hermet close stu...


In [36]:
#Lemmatization
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

df['tokens'] = df['tokens'].apply(lambda words: [lemmatizer.lemmatize(w) for w in words])
df.head()

,text,reviews.rating,label,tokens,processed_text
0,i initially had trouble deciding between the p...,5.0,1,"[initi, troubl, decid, paperwhit, voyag, revie...",initi troubl decid paperwhit voyag review less...
1,allow me to preface this with a little history...,5.0,1,"[allow, prefac, littl, histori, casual, reader...",allow prefac littl histori casual reader own n...
2,i am enjoying it so far great for reading had ...,4.0,1,"[enjoy, far, great, read, origin, fire, sinc, ...",enjoy far great read origin fire sinc 2012 fir...
3,i bought one of the first paperwhites and have...,5.0,1,"[bought, one, first, paperwhit, plea, constant...",bought one first paperwhit plea constant compa...
4,i have to say upfront i dont like coroporate ...,5.0,1,"[say, upfront, dont, like, coropor, hermet, cl...",say upfront dont like coropor hermet close stu...


In [24]:
#Join Words
df['processed_text'] = df['tokens'].apply(lambda words: ' '.join(words))

In [25]:
#Step 9: TF-IDF Vectorization
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
X = tfidf.fit_transform(df['processed_text'])

y = df['label']

In [26]:
#Step 10: Train-Test Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [27]:
#Step 11: Model Training
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()
model.fit(X_train, y_train)

MultinomialNB()

In [28]:
#Step 12: Prediction
y_pred = model.predict(X_test)

In [29]:
#Step 13: Evaluation
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.9558011049723757
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         8
           1       0.96      1.00      0.98       173

    accuracy                           0.96       181
   macro avg       0.48      0.50      0.49       181
weighted avg       0.91      0.96      0.93       181



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
